# 2. Nettoyage et chargement (ETL)

- **Extract** : lire les deux CSV
- **Transform** : corriger les types, traduire les codes, garder les colonnes utiles
- **Load** : écrire le résultat dans une base **SQLite**

In [6]:
import sqlite3
import pandas as pd

caract = pd.read_csv("../data/raw/caract-2024.csv", sep=";")
usagers = pd.read_csv("../data/raw/usagers-2024.csv", sep=";")

## Corriger les types
- `lat`/`long` : la virgule est remplacée par un point, puis la valeur est convertie en nombre décimal.
- `heure` : les deux premiers caractères de `hrmn` (`"17:30"` → 17).
- `date` et `jour_semaine` : construits à partir de `an`, `mois` et `jour`.

In [7]:
# lat/long : virgule décimale -> point, puis conversion en nombre décimal (float)
caract["lat"] = caract["lat"].str.replace(",", ".").astype(float)
caract["long"] = caract["long"].str.replace(",", ".").astype(float)

# hrmn vaut par exemple "17:30" : on garde les 2 premiers caractères -> l'heure (17)
caract["heure"] = caract["hrmn"].str[:2].astype(int)

# On construit une vraie date à partir des colonnes an / mois / jour
caract["date"] = pd.to_datetime(
    caract[["an", "mois", "jour"]].rename(columns={"an": "year", "mois": "month", "jour": "day"})
)

# Jour de la semaine : dayofweek donne 0 pour lundi ... 6 pour dimanche
jours = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
caract["jour_semaine"] = caract["date"].dt.dayofweek.map(lambda i: jours[i])

## Traduire les codes en libellés
Les dictionnaires sont construits à partir de la documentation officielle. `.map()` remplace chaque code par son libellé.

Un code absent du dictionnaire, comme `-1`, devient une **valeur vide (NaN)**, et c'est voulu : un NaN est **ignoré** dans les calculs, alors qu'un `-1` les fausserait.
Exemple : la moyenne de 20, 40 et −1 vaut 19,7, alors que celle de 20, 40 et NaN vaut 30.

J'ajoute aussi une colonne `periode` simplifiée (Jour / Crépuscule / Nuit) pour rendre les graphiques plus lisibles.

In [8]:
# Un code absent du dictionnaire (comme -1) devient une valeur vide (NaN) :
# les "non renseigné" deviennent ainsi de vraies valeurs manquantes.
LUM = {1: "Plein jour", 2: "Crépuscule ou aube", 3: "Nuit sans éclairage",
       4: "Nuit, éclairage non allumé", 5: "Nuit avec éclairage"}
AGG = {1: "Hors agglomération", 2: "En agglomération"}
ATM = {1: "Normale", 2: "Pluie légère", 3: "Pluie forte", 4: "Neige / grêle",
       5: "Brouillard / fumée", 6: "Vent fort / tempête", 7: "Temps éblouissant",
       8: "Temps couvert", 9: "Autre"}
GRAV = {1: "Indemne", 2: "Tué", 3: "Blessé hospitalisé", 4: "Blessé léger"}
CATU = {1: "Conducteur", 2: "Passager", 3: "Piéton"}
SEXE = {1: "Homme", 2: "Femme"}

caract["luminosite"] = caract["lum"].map(LUM)
caract["agglomeration"] = caract["agg"].map(AGG)
caract["meteo"] = caract["atm"].map(ATM)
# Version simplifiée de la luminosité en 3 catégories, plus lisible dans les graphiques
caract["periode"] = caract["lum"].map({1: "Jour", 2: "Crépuscule", 3: "Nuit", 4: "Nuit", 5: "Nuit"})

usagers["gravite"] = usagers["grav"].map(GRAV)
usagers["categorie"] = usagers["catu"].map(CATU)
usagers["sexe_lib"] = usagers["sexe"].map(SEXE)
usagers["age"] = 2024 - usagers["an_nais"]

## Sélection des colonnes et contrôles qualité

In [9]:
accidents = caract[["Num_Acc", "date", "mois", "jour_semaine", "heure", "dep", "com",
                    "agglomeration", "luminosite", "periode", "meteo", "lat", "long"]]
usagers_clean = usagers[["Num_Acc", "categorie", "gravite", "sexe_lib", "age"]]

# Contrôles : valeurs vides par colonne, et ordres de grandeur
print(accidents.isna().sum())
print(usagers_clean.isna().sum())
accidents[["heure", "lat", "long"]].describe()

Num_Acc          0
date             0
mois             0
jour_semaine     0
heure            0
dep              0
com              0
agglomeration    0
luminosite       0
periode          0
meteo            0
lat              0
long             0
dtype: int64
Num_Acc         0
categorie       0
gravite         0
sexe_lib     2395
age          2579
dtype: int64


,heure,lat,long
count,54402.000000,54402.000000,54402.000000
mean,13.358774,44.051229,1.240096
std,5.638434,12.665655,19.460609
min,0.000000,-22.433239,-178.094387
25%,9.000000,44.519282,1.153567
50%,14.000000,47.717121,2.392034
75%,18.000000,48.861732,4.732405
max,23.000000,51.078740,167.863219


- **Accidents :** aucune valeur manquante.
- **Usagers :** sexe inconnu pour 2 395 personnes et âge inconnu pour 2 579 (environ 2 %), ce qui reste acceptable.
- **Coordonnées :** `lat` va de −22 à 51 et `long` de −178 à 168. Ce ne sont **pas des erreurs** : ce sont les territoires d'outre-mer (La Réunion, Nouvelle-Calédonie, Wallis-et-Futuna). Je les garde dans les chiffres, et je limite la carte du dashboard à la métropole pour qu'elle reste lisible.

## Chargement dans SQLite
Deux tables, reliées par `Num_Acc` :
- `accidents` : une ligne par accident
- `usagers` : une ligne par personne impliquée

In [10]:
con = sqlite3.connect("../data/accidents.db")   # crée le fichier de base s'il n'existe pas
accidents.to_sql("accidents", con, if_exists="replace", index=False)
usagers_clean.to_sql("usagers", con, if_exists="replace", index=False)

# Vérification avec notre première requête SQL
pd.read_sql("SELECT COUNT(*) AS nb_accidents FROM accidents", con)

,nb_accidents
0,54402


✅ Les 54 402 accidents sont bien chargés dans la base.